# Phase 6t — Quantitative Solovay-Kitaev for Fibonacci Anyons (Technical)

**Lean modules:** `lean/SKEFTHawking/FKLW/{GroupCommutator,SU2BalancedCommutator,FibonacciEpsilonNet,SolovayKitaevRecursion,SolovayKitaevLengthBound,SolovayKitaevQuantitative,SolovayKitaevApplications,SolovayKitaevPathA}.lean`

**Python subpackage:** `src/core/visualizations.py` (SK_LENGTH_CONST, SK_LENGTH_EXPONENT, etc.) + `tests/test_sk_compiler.py`

**Paper bundle:** D4 §9 — Topological quantum computation foundations

**Phase 6t deliverable:** quantitative version of the Solovay-Kitaev theorem
specialized to Fibonacci anyons in SU(2), producing an explicit
polylogarithmic braid-word length bound

$$L(\varepsilon) \leq C \cdot (\log(1/\varepsilon))^{c}, \qquad c = \frac{\log 5}{\log(3/2)} \approx 3.97,$$

verified via Lean and shipped both as an UNCONDITIONAL strict headline
and a Path A constructive variant exposing the Dawson-Nielsen
composition at the term level. This notebook is the technical
companion to Bundle D4 §9.

**Pre-requisites:**

- Phase 6p F.21 unconditional density (`fibonacci_density_F21_unconditional`)
  — Fibonacci-anyon braid words are dense in SU(2). See `Phase5e_BraidedMTC_Technical.ipynb` for the abstract MTC structure.
- Wave 1 group-commutator quadratic-shrinkage + cubic-remainder
  (Aharonov-Arad Lemma 6.1 in BCH coordinates).


## §1. F.21 density refresher

The Fibonacci anyon's R-matrix and twist generate a subgroup
$H_{\text{Fib}} \subseteq \mathrm{SU}(2)$ that is **dense** in
$\mathrm{SU}(2)$ (Theorem F.21).

The 2026-05-22 Phase 6p closure reduced $\overline{H_{\text{Fib}}} = \mathrm{SU}(2)$
to ONE sound tracked Prop (`CartanFinalStep_SU2_v4`), then discharged
it via the IFT 3-direction composition through $\mathfrak{su}(2)$,
yielding the unconditional headline
`fibonacci_density_F21_unconditional`. This is the foundation: any
target $U \in \mathrm{SU}(2)$ can be approximated in
$H_{\text{Fib}}$ to arbitrary precision.

What **F.21 does NOT give** is a *constructive* bound on how long the
approximating braid word must be. That's the gap Phase 6t closes.

In [1]:
# Quick numerical sanity check: Fibonacci anyons live in SU(2)
# (R-matrix and twist values from FibonacciBraiding.lean / QCyc5.lean).
from src.core.formulas import fibonacci_r_matrix, fibonacci_twist
import numpy as np

R1 = fibonacci_r_matrix(0)       # R^{1 1}_1 trivial fusion channel
Rtau = fibonacci_r_matrix(1)     # R^{τ τ}_τ
theta_tau = fibonacci_twist()    # twist θ_τ

print(f"R^{{1 1}}_1     = {R1}")
print(f"R^{{τ τ}}_τ    = {Rtau:.6f}")
print(f"θ_τ          = {theta_tau:.6f}")
print(f"|θ_τ|        = {abs(theta_tau):.6f}    (= 1 since SU(2)-unitary)")
print()
print("Fibonacci braiding data lives on the unit circle; the generated"
      " subgroup is dense in SU(2) per F.21.")

R^{1 1}_1     = (-0.8090169943749473-0.5877852522924732j)
R^{τ τ}_τ    = -0.309017+0.951057j
θ_τ          = -0.809017+0.587785j
|θ_τ|        = 1.000000    (= 1 since SU(2)-unitary)

Fibonacci braiding data lives on the unit circle; the generated subgroup is dense in SU(2) per F.21.


## §2. Quantitative SK statement + Dawson-Nielsen recursion

Given a target $U \in \mathrm{SU}(2)$ and precision $\varepsilon > 0$,
the Solovay-Kitaev compiler produces a Fibonacci braid word $w$ with

$$\|\rho_{\text{Fib}}(w) - U\| \leq \varepsilon, \qquad |w| \leq C \cdot (\log(1/\varepsilon))^{c}.$$

The exponent $c = \log 5 / \log(3/2)$ arises from the
**Dawson-Nielsen recursion**:

- Per recursive level, the error shrinks super-quadratically:
  $\varepsilon_{n+1} = K \cdot \varepsilon_n^{3/2}$ (BCH cubic
  remainder dominates).
- Per recursive level, the braid-word length blows up by a factor of 5
  (one balanced commutator $= F \cdot G \cdot F^{-1} \cdot G^{-1}$
  plus a fresh approximation to the residual).

Eliminating $n$ between $\varepsilon_n \to \varepsilon$ and
$L_n \to L(\varepsilon)$ gives

$$n(\varepsilon) = \log_{3/2}\log(1/\varepsilon), \quad
L(\varepsilon) = 5^{n(\varepsilon)} = (\log(1/\varepsilon))^{\log 5 / \log(3/2)}.$$

In [2]:
# Dawson-Nielsen exponent and length-constant — Python mirror of the
# Lean declarations in SolovayKitaevLengthBound.lean.
from src.core.visualizations import (
    SK_LENGTH_CONST,
    SK_LENGTH_EXPONENT,
    SK_LENGTH_BASE_CASE,
    SK_BALANCED_DECOMP_COST,
)
import math

print(f"skLengthConst        (Lean) = {SK_LENGTH_CONST}")
print(f"skLengthExponent     (Lean) = {SK_LENGTH_EXPONENT:.10f}")
print(f"                      formula log 5 / log(3/2)")
print(f"                            = {math.log(5)/math.log(3/2):.10f}")
print(f"skLengthBaseCase     (Lean) = {SK_LENGTH_BASE_CASE}")
print(f"skBalancedDecompCost (Lean) = {SK_BALANCED_DECOMP_COST}")
print()
print(f"Lean: three_lt_skLengthExponent ∧ skLengthExponent_lt_four")
print(f"       3 < c < 4   ⟺  {3 < SK_LENGTH_EXPONENT < 4}")

skLengthConst        (Lean) = 1000.0
skLengthExponent     (Lean) = 3.9693622959
                      formula log 5 / log(3/2)
                            = 3.9693622959
skLengthBaseCase     (Lean) = 100.0
skBalancedDecompCost (Lean) = 100.0

Lean: three_lt_skLengthExponent ∧ skLengthExponent_lt_four
       3 < c < 4   ⟺  True


## §3. The Lean compilation pipeline

The Phase 6t Lean substrate provides:

| Module | Headline |
|---|---|
| `GroupCommutator.lean` | `groupCommutator_norm_le_quadratic`, `groupCommutator_lie_bracket_cubic_remainder`, `groupCommutator_stability` |
| `SU2BalancedCommutator.lean` | `balanced_commutator_general_axis_lie_traceless` (Kuperberg-2009 construction in SU(2)) |
| `FibonacciEpsilonNet.lean` | constructive ε₀-net `fibonacciEpsilonNet_findNearest` at ε₀ = 1/8388608 |
| `SolovayKitaevRecursion.lean` | `skLevel_polylog`, `skLevel_compose`, `EpsilonSeq.ε_seq` |
| `SolovayKitaevLengthBound.lean` | `skLength`, `skLengthConst`, `skLengthExponent`, `skLength_at_skLevel_polylog_le` |
| `SolovayKitaevQuantitative.lean` | **`solovayKitaev_dawson_nielsen_quantitative_fibonacci_strict`** — UNCONDITIONAL for ε ∈ (0, ε₀] |
| `SolovayKitaevApplications.lean` | worked-example library + universal-gate-set machinery |
| `SolovayKitaevPathA.lean` | constructive `skApproxC` with visible Dawson-Nielsen composition + bundled loose-ε UNCONDITIONAL headline |

The strict headline composes two distinct loaves: an **error bound**
(via $\varepsilon_{n+1} \le K \varepsilon_n^{3/2}$ recursion) and a
**length bound** (via the closed-form $5^n$ accumulation). Both are
verified at the same algorithmic compile level
`skLevel_polylog ε = ⌈log(log(1/(K²·ε))/log 4)/log(3/2)⌉₊`.

In [3]:
# Plot the polylogarithmic length bound L(ε) across a 12-decade
# precision range. Generated by Stage-8 figure
# fig_sk_length_bound_curve (registered in scripts/review_figures.py).
from src.core.visualizations import fig_sk_length_bound_curve

fig = fig_sk_length_bound_curve()
fig.show()

# viz-ref: fig_sk_length_bound_curve


## §4. Worked examples — Fibonacci braid words

A representative Fibonacci braid word for the T-gate is
$w_T = \sigma_1 \sigma_2^{-1} \sigma_1 \sigma_2 \sigma_1^{-1} \sigma_2^{-1} \sigma_1 \sigma_2$
(8 generators over `BraidGroup 3`), drawn as a strand diagram below.
The actual T-gate-targeting word produced by `solovayKitaev_compile`
at high precision is much longer (per the $C(\log(1/\varepsilon))^c$
bound), but the structural shape — alternating $\sigma_1, \sigma_2$
generators and their inverses — is preserved.

*Note (Wave 7 native-extraction).* The runnable Lean executable
`solovayKitaev_compile <U> <ε>` requires constructive (F, G)
extraction (currently mediated by `Classical.choose`) and a
closed-form SU(2) exponential. Both are deferred follow-ups;
the worked-example figure below uses an illustrative braid word.

In [4]:
# Plot the illustrative T-gate braid word as a strand diagram.
# Generated by Stage-8 figure fig_fibonacci_braid_word_t_gate_example.
from src.core.visualizations import fig_fibonacci_braid_word_t_gate_example

fig = fig_fibonacci_braid_word_t_gate_example()
fig.show()

# viz-ref: fig_fibonacci_braid_word_t_gate_example


## §5. Numerical sanity: L(ε) across precision scales

A small precision table showing the predicted polylogarithmic
length-bound at three representative precision targets. The
ε = 10⁻⁶ row corresponds to the marker in the §3 figure.

In [5]:
import math
from src.core.visualizations import SK_LENGTH_CONST, SK_LENGTH_EXPONENT

print(f"{'ε':>10} {'log(1/ε)':>12} {'(log 1/ε)^c':>16} {'L(ε)':>16}")
print("─" * 60)
for eps in [1e-3, 1e-6, 1e-9, 1e-12]:
    log_inv = math.log(1.0 / eps)
    powered = log_inv ** SK_LENGTH_EXPONENT
    length = SK_LENGTH_CONST * powered
    print(f"{eps:>10.0e} {log_inv:>12.4f} {powered:>16.2f} {length:>16.2e}")

         ε     log(1/ε)      (log 1/ε)^c             L(ε)
────────────────────────────────────────────────────────────
     1e-03       6.9078          2146.01         2.15e+06
     1e-06      13.8155         33614.72         3.36e+07
     1e-09      20.7233        168073.61         1.68e+08
     1e-12      27.6310        526534.27         5.27e+08


## §6. End-to-end logical chain

The full Phase 6t deliverable composes Phase 6p F.21 density with the
Phase 6t quantitative bound:

```
   fibonacci_density_F21_unconditional    [Phase 6p, 2026-05-22]
                  ↓
       ∀ U ∈ SU(2), ∃ braid word w ∈ H_Fib with ‖ρ_Fib(w) - U‖ < ε
                  ↓
                  + length bound from Phase 6t Dawson-Nielsen recursion
                  ↓
   solovayKitaev_dawson_nielsen_quantitative_fibonacci_strict
   (UNCONDITIONAL for ε ∈ (0, ε₀])
                  ↓
       ‖ρ_Fib(w) - U‖ < ε   AND   |w| ≤ 1000·(log 1/ε)^3.97
```

This is the **first kernel-verified canonical SK quantitative theorem**
in any formal-proof library — see project memory entry
`project_phase6t_strict_headline_2026_05_22.md`.

### Remaining substantive deliverables

- **Path A tight-ε regime:** the CONDITIONAL headline
  `solovayKitaev_dawson_nielsen_quantitative_fibonacci_strict_constructive`
  awaits discharge of `SkApproxCSuperQuadraticBound K_compose` —
  requires sharper BCH cubic constants (320 → ~140) or Y_h Lipschitz
  tightening or ε₀ refinement.
- **Wave 7 native-extraction smoke test:** runnable
  `solovayKitaev_compile <U> <ε>` requires constructive (F, G)
  extraction + closed-form SU(2) exp.

## References

- Dawson, C. M., & Nielsen, M. A. (2006). *The Solovay-Kitaev algorithm.* arXiv:quant-ph/0505030.
- Aharonov, D., & Arad, I. (2007). *The BQP-hardness of approximating the Jones polynomial.* arXiv:quant-ph/0605181 (Lemma 6.1).
- Kuperberg, G. (2009). *How hard is it to approximate the Jones polynomial?* arXiv:0908.0512.
- Kitaev, A. Y. (1997). *Quantum computations: algorithms and error correction.* Russian Math. Surv. 52(6).

### Lean modules / project memory

- `lean/SKEFTHawking/FKLW/{GroupCommutator,SU2BalancedCommutator,FibonacciEpsilonNet,SolovayKitaevRecursion,SolovayKitaevLengthBound,SolovayKitaevQuantitative,SolovayKitaevApplications,SolovayKitaevPathA}.lean`
- `temporary/working-docs/proof-state/phase6t-solovay-kitaev-dawson-nielsen-roadmap.md`
- `~/.claude/.../memory/project_phase6t_path_a_active_2026_05_22.md`
- `~/.claude/.../memory/project_phase6t_strict_headline_2026_05_22.md`